In [ ]:
import time
from model_cnn_transformer import *
from pathlib import Path
import torch, os
import torch.nn as nn
from torch.utils.data import DataLoader, random_split

torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark = True # faster convolutions on fixed input sizes

BASE_DIR = Path(os.getcwd())

IMG_DIR = BASE_DIR / "dataset" / "images"
CSV_PATH = BASE_DIR / "dataset" / "labels.csv"
SAVE_DIR = Path("saved_models")
SAVE_DIR.mkdir(exist_ok=True)
CHECKPOINT_PATH = SAVE_DIR / "cnn_transformer_ctc_best.pt"

BATCH_SIZE = 128
EPOCHS = 20
LR = 3e-5 # learning rate
WEIGHT_DECAY = 2e-2 # L2 regularization strength for AdamW
NUM_WORKERS = 2 # DataLoader worker processes
WARMUP_EPOCHS = 4 # number of epochs for linear LR warmup

NUM_CLASSES = len(ALPHABET) + 1 # vocabulary size + CTC blank token

In [ ]:
ds = OCRDataset(IMG_DIR, CSV_PATH)
len(ds)

n = len(ds)
test_size = int(0.1 * n) # 10% for test
val_size = int(0.1 * n) # 10% for validation
train_size = n - val_size - test_size # 80% for training

train_ds, val_ds, test_ds = random_split(
    ds, [train_size, val_size, test_size]
)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=(DEVICE=="cuda"),
    persistent_workers=(NUM_WORKERS > 0),
    collate_fn=collate_fn
)

val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=(DEVICE=="cuda"),
    persistent_workers=(NUM_WORKERS > 0),
    collate_fn=collate_fn
)

test_loader = DataLoader(
    test_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=(DEVICE=="cuda"),
    persistent_workers=(NUM_WORKERS > 0),
    collate_fn=collate_fn
)

model = CNNTransformerOCR(NUM_CLASSES).to(DEVICE)

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

def get_lr(epoch: int) -> float:
    """
    Learning rate schedule:
      1. Linear warmup for the first WARMUP_EPOCHS epochs
      2. Cosine decay down to 5% of peak LR for the remaining epochs
    """
    if epoch < WARMUP_EPOCHS:
        return LR * (epoch + 1) / WARMUP_EPOCHS
    t = (epoch - WARMUP_EPOCHS) / max(1, (EPOCHS - WARMUP_EPOCHS))
    cos = 0.5 * (1 + math.cos(math.pi * t))
    return LR * (0.05 + (1 - 0.05) * cos)

# CTC loss: blank index 0, zero_infinity ignores inf losses (numerical stability)
ctc_loss = nn.CTCLoss(blank=0, zero_infinity=True)

use_amp = (DEVICE == "cuda")
scaler = torch.amp.GradScaler("cuda", enabled=use_amp) # prevents gradient underflow in float16

@torch.no_grad()
def evaluate(model, loader):
    # evaluates the model on a DataLoader and returns average loss and CER
    model.eval()
    total_loss = 0.0
    total_chars = 0
    total_cer = 0.0

    for x, y_cat, y_lens, texts in loader:
        x = x.to(DEVICE, non_blocking=True)
        y_cat = y_cat.to(DEVICE, non_blocking=True)
        y_lens = y_lens.to(DEVICE, non_blocking=True)

        logits = model(x)  # [T,B,C]
        log_probs = logits.log_softmax(dim=-1)

        T, B, C = log_probs.shape
        input_lens = torch.full((B,), T, dtype=torch.long, device=DEVICE)

        loss = ctc_loss(log_probs, y_cat, input_lens, y_lens)
        total_loss += loss.item() * B

       # get Character Error Rate for each sample in the batch
        preds = ctc_greedy_decode(logits)
        for p, gt in zip(preds, texts):
            total_cer += cer(p, gt) * len(gt)
            total_chars += len(gt)

    return total_loss / len(loader.dataset), (total_cer / max(1, total_chars))

def train_one_epoch(model, loader, epoch):
    model.train()
    total_loss = 0.0

    # Update learning rate for this epoch
    lr = get_lr(epoch)
    for pg in optimizer.param_groups:
        pg["lr"] = lr

    for x, y_cat, y_lens, _texts in loader:
        x = x.to(DEVICE, non_blocking=True)
        y_cat = y_cat.to(DEVICE, non_blocking=True)
        y_lens = y_lens.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        # Forward pass with AMP (float16)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
            logits = model(x)
            log_probs = logits.log_softmax(dim=-1)
            T, B, C = log_probs.shape
            input_lens = torch.full((B,), T, dtype=torch.long, device=DEVICE)
            loss = ctc_loss(log_probs, y_cat, input_lens, y_lens)

        # Backward pass with gradient scaling (prevents underflow in float16)
        scaler.scale(loss).backward()

        # Unscale before clipping, then clip gradients to prevent exploding gradients
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)

        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * x.size(0)

    return total_loss / len(loader.dataset), lr

In [ ]:
best_val_cer = 1e9
best_epoch = -1

start_time = time.time()

for epoch in range(EPOCHS):
    tr_loss, lr = train_one_epoch(model, train_loader, epoch)
    va_loss, va_cer = evaluate(model, val_loader)

    print(f"Epoch {epoch+1:02d}/{EPOCHS} | lr={lr:.6f} | train_loss={tr_loss:.4f} | val_loss={va_loss:.4f} | val_CER={va_cer:.4f}")

    # Save checkpoint if validation CER improved
    improved = va_cer < best_val_cer - 1e-6
    if improved:
        best_val_cer = va_cer
        best_epoch = epoch + 1

        torch.save({
            "model": model.state_dict(),
            "alphabet": ALPHABET,
            "img_w": IMG_W,
            "img_h": IMG_H,
            "best_val_cer": best_val_cer,
            "epoch": best_epoch,
        }, CHECKPOINT_PATH)

total_time = time.time() - start_time
print(f"\nBest val CER: {best_val_cer:.6f} at epoch {best_epoch}")
print(f"Total training time: {total_time/60:.2f} min")

ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
model.load_state_dict(ckpt["model"])

te_loss, te_cer = evaluate(model, test_loader)
print(f"TEST | loss={te_loss:.4f} | CER={te_cer:.4f}")